In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pathlib import Path
from tqdm.auto import tqdm                    # NEW ✨

# ----------------------- 1. CONFIG -----------------------
model_name   = "facebook/nllb-200-distilled-600M"
src_lang     = "eng_Latn"
tgt_lang     = "npi_Deva"
input_file   = Path("captions.txt")     # <- local file, adjust path
output_file  = Path("captions_ne.txt")
batch_size   = 32

# ----------------------- 2. SETUP ------------------------
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

tokenizer.src_lang = src_lang
forced_bos_id      = tokenizer.convert_tokens_to_ids(tgt_lang)

# ----------------------- 3. LOAD DATA --------------------
with input_file.open(encoding="utf-8") as f:
    captions = [line.strip() for line in f if line.strip()]

total_lines = len(captions)

# ----------------------- 4. TRANSLATE --------------------
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]

nepali_captions = []
model.eval()

with torch.no_grad():
    for batch in tqdm(chunks(captions, batch_size),
                      total=(total_lines + batch_size - 1) // batch_size,
                      desc="Translating",
                      unit="batch"):
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(device)
        generated = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_id,
            max_length=30
        )
        nepali_captions.extend(
            tokenizer.batch_decode(generated, skip_special_tokens=True)
        )

# ----------------------- 5. SAVE OUTPUT ------------------
with output_file.open("w", encoding="utf-8") as f:
    for line in nepali_captions:
        f.write(line + "\n")

print(f"✅  Translated {total_lines} captions → {output_file.resolve()}")
